## HW 5: Principal Component Analysis

In this HW assignment, we will walk through an example of using Principal Component Analysis (PCA) on a dataset involving [iris plants](https://en.wikipedia.org/wiki/Iris_plant).




# Principal Component Analysis (PCA)

Principal Component Analysis (PCA) is a dimensionality reduction technique that transforms a dataset's features into a new coordinate system.

Instead of describing the data using the original features, PCA finds **new orthogonal axes (principal components)** that:

- Capture the **maximum variance** in the data
- Are **uncorrelated** with each other
- Are ordered by importance (variance explained)

---

## Why use PCA?

- Reduce dimensionality while preserving structure
- Remove correlated features
- Improve visualization (e.g., projecting high-D data to 2D or 3D)
- Preprocess data for machine learning models

---

## Intuition

PCA finds the directions in which the data varies the most.

- The **first principal component** is the direction of maximum variance
- The **second principal component** is the next largest variance, orthogonal to the first
- And so on...

## Step 1: Centering the Data

Given a dataset $X \in \mathbb{R}^{n \times d}$:

We first subtract the mean of each feature:

$$
X_{\text{centered}} = X - \mu
$$

where:

$$
\mu = \frac{1}{n} \sum_{i=1}^{n} X_i
$$

---

## Step 2: Covariance Matrix

The covariance matrix describes how features vary together:

$$
C = \frac{1}{n-1} X_{\text{centered}}^T X_{\text{centered}}
$$

- Diagonal elements → variances of each feature
- Off-diagonal elements → correlations between features

## Step 3: Eigenvalue Decomposition

PCA is based on solving the eigenvalue problem:

$$
C \mathbf{v} = \lambda \mathbf{v}
$$

where:

- $\mathbf{v}$ = eigenvectors (principal directions)
- $\lambda$ = eigenvalues (variance along each direction)

---

## Interpretation

- Each eigenvector defines a **principal component**
- Each eigenvalue tells us **how much variance is explained by that component**

We sort eigenvalues in descending order:

$$
\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_d
$$

## Step 4: Projection

To reduce dimensionality, we project the data onto the top $k$ eigenvectors:

$$
Z = X_{\text{centered}} W_k
$$

where:

- $W_k = [\mathbf{v}_1, \mathbf{v}_2, ..., \mathbf{v}_k]$
- $Z$ is the transformed data in $k$ dimensions

---

## Key Idea

We replace the original features with:

- Linear combinations of features
- That capture the most important variation in the dataset

## Explained Variance

The fraction of variance explained by each component is:

$$
\text{Explained Variance Ratio} = \frac{\lambda_i}{\sum_{j=1}^{d} \lambda_j}
$$

---

## Cumulative Explained Variance

We often choose $k$ such that:

$$
\sum_{i=1}^{k} \frac{\lambda_i}{\sum_{j=1}^{d} \lambda_j} \approx 0.9 \ \text{or} \ 0.95
$$

This ensures we retain most of the information in fewer dimensions.

## PCA via Singular Value Decomposition (SVD) (IMPORTANT)

Instead of explicitly forming the covariance matrix, our preferred approach to PCA can be computed using SVD:

$$
X_{\text{centered}} = U \Sigma V^T
$$

- Columns of $V$ = principal components
- Singular values relate to eigenvalues:

$$
\lambda_i = \frac{\sigma_i^2}{n-1}
$$

---

## Why SVD?

- More numerically stable
- Preferred for large datasets
- Used in most ML libraries (e.g., scikit-learn)

In [ ]:
%pip install --quiet iwut
%load_ext iwut
%wut on

Note: you may need to restart the kernel to use updated packages.


In [2]:
from sklearn.datasets import load_iris
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

To begin, run the following cell to load the dataset into this notebook. 
* `iris_features` will contain a numpy array of 4 attributes for 150 different plants (shape `150 x 4`). 
* `iris_target` will contain the class of each plant. There are 3 classes of plants in the dataset: Iris-Setosa, Iris-Versicolour, and Iris-Virginica. The class names will be stored in `iris_target_names`.
* `iris_feature_names` will be a list of 4 names, one for each attribute in `iris_features`. 

In [3]:
from sklearn.datasets import load_iris
iris_data = load_iris() # Loading the dataset

# Unpacking the data into arrays
iris_features = iris_data['data']
iris_target = iris_data['target']
iris_feature_names = iris_data['feature_names']
iris_target_names = iris_data['target_names']

# Convert iris_target to string labels instead of int labels currently (0, 1, 2) for the classes
iris_target = iris_target_names[iris_target]

Let's explore the data by creating a scatter matrix of our iris features. To do this, we'll create 2D scatter plots for every possible pair of our four features. This should result in six total scatter plots in our scatter matrix with the classes labeled in distinct colors for each plot.

In [ ]:
# Run this cell to see the plot; no further action is needed.
fig = plt.figure(figsize=(14, 10))
plt.suptitle("Scatter Matrix of Iris Features", fontsize=20)
plt.subplots_adjust(wspace=0.3, hspace=0.3)
for i in range(1, 4):
    for j in range(i):
        plot_index = 3*j + i
        plt.subplot(3, 3, plot_index)
        sns.scatterplot(x=iris_features[:, i],
                        y=iris_features[:, j],
                        hue=iris_target,
                       legend=(plot_index == 1))
        plt.xlabel(iris_feature_names[i])
        plt.ylabel(iris_feature_names[j])
        if plot_index == 1:
            plt.legend().remove()

# same legend for all subplots.
fig.legend(loc='lower left') 
fig.tight_layout()
plt.show()

## Question 1a

To apply PCA, we will first need to center and scale the data so that the mean of each feature is 0, and the standard deviation of each feature is 1. 

Compute the columnwise mean of `iris_features` in the cell below and store it in `iris_mean`, and compute the columnwise standard deviation of `iris_features` and store it in `iris_std`. Each should be a numpy array of 4 means, 1 for each feature. Then, subtract `iris_mean` from `iris_features` and divide by `iris_std`, and finally, save the result in `features`.

**Hints:** 
* Use `np.mean` or `np.average` to compute `iris_mean`, and pay attention to the `axis` argument.
* If you are confused about how numpy deals with arithmetic operations between arrays of different shapes, see this note about [broadcasting](https://docs.scipy.org/doc/numpy/user/basics.broadcasting.html) for explanations/examples.

<!--
BEGIN QUESTION
name: q1a
-->

In [ ]:
iris_mean = ...
iris_std = ...
features = ...
iris_mean, iris_std

## Question 1b

PCA is a specific application of the singular value decomposition (SVD) for matrices. In the following cell, let's use the [`np.linalg.svd`](https://docs.scipy.org/doc/numpy/reference/generated/numpy.linalg.svd.html) function to compute the SVD of our `features` matrix. Store the left singular vectors, singular values, and right singular vectors in `u`, `s`, and `vt`, respectively. Note that `vt` corresponds to $V^T$. Set the `full_matrices` argument of `np.linalg.svd` to `False`.



In [ ]:
u, s, vt = ...
print(f"Dimensions of U: {u.shape}")
print(f"1D List of diagonal elements of Sigma: {s}")
print(f"Dimensions of V Transpose: {vt.shape}")

## Question 1c

What can we learn from the singular values in `s`? From a technical standpoint, we can measure the amount of variance captured by the i'th principal component as:

$\sigma_i^2/N$, where $\sigma_i$ is the singular value of the i'th principal component and $N$ is the total number of data points.

Compute the total variance of our data below by summing the square of each singular value in `s` and dividing the result by the total number of data points. Store the result in the variable `total_variance`.

<!--
BEGIN QUESTION
name: q1c
-->

In [ ]:
iris_total_variance = ...

print("iris_total_variance: {:.3f} should approximately equal the sum of the feature variances: {:.3f}"
      .format(iris_total_variance, np.sum(np.var(features, axis=0))))

As you can see, `total_variance` is equal to the sum of the feature variances.

## Question 2a

Let's now use only the first two principal components to see what a 2D version of our iris data looks like.

First, construct the 2D version of the iris data by multiplying our `features` array with the first two right singular vectors in `v`. Because the first two right singular vectors are directions for the first two principal components, this will project the iris data down from a 4D subspace to a 2D subspace.

**Hints:**
* To matrix-multiply two numpy arrays, use `@` or `np.dot`.
* Note that the output of `np.linalg.svd` is `vt` and not `v`: the first two right singular vectors in `v` will be the first two columns of `v`, or the first two rows of `vt` (transposed to be column vectors instead of row vectors). 
* Since we want to obtain a 2D version of our iris dataset, the shape of `iris_2d` should be (150, 2).

<!--
BEGIN QUESTION
name: q2a
-->

In [ ]:
features.shape, vt.shape

In [ ]:
iris_2d = ...
iris_2d.shape

Now, run the cell below to create the scatter plot of our 2D version of the iris data, `iris_2d`.

In [ ]:
# Run this cell to see the plot; no further action is needed.
plt.figure(figsize = (7, 7))
plt.title("PC2 vs. PC1 for Iris Data")
plt.xlabel("Iris PC1")
plt.ylabel("Iris PC2")
sns.scatterplot(x = iris_2d[:, 0], y = iris_2d[:, 1], hue = iris_target);

## Question 2b

What do you observe about the plot above? If you were given a point in the subspace defined by PC1 and PC2, how well would you be able to classify the point as one of the three iris types?

<!--
BEGIN QUESTION
name: q2b
-->

_Type your answer here, replacing this text._

## Question 2c

What proportion of the total variance is accounted for when we project the iris data down to two dimensions? Compute this quantity in the cell below by dividing the variance captured by the first two singular values (also known as component scores) in `s` by the `total_variance` you calculated previously. Store the result in `two_dim_variance`.

<!--
BEGIN QUESTION
name: q2c
-->

In [ ]:
iris_2d_variance = ...
iris_2d_variance

You should see a value of ~95%. Most of the variance in the data is explained by the two-dimensional projection.

## Question 3

As a last step, we will create a [scree plot](https://en.wikipedia.org/wiki/Scree_plot) to visualize the weight of each principal component. In the cell below, create a scree plot by creating a line plot of the component scores (variance captured by each principal component) vs. the principal component number (1st, 2nd, 3rd, or 4th). Your graph should match the image below:

***Hint***: You may find `plt.xticks()` helpful when formatting your plot axes. See question 1c if you need help remembering how to get the variance of a specific principal component.



In [ ]:
plt.xticks(...)
plt.xlabel("Principal Component")
plt.ylabel("Variance (Component Scores)")
plt.title("Scree Plot of Iris Principal Components")
plt.plot(...)